# 改写检索问题

用户的说法和书里的说法不一致时，可以调整检索文字，但不能改变问题本身。下面分别验证三种做法：去掉口语歧义、把两部分问题拆开，以及在细节问题缺少背景时先查一般原理。每个例子都把回答要点、资料量和检索次数一起记下来。


## 问题改写的原理与边界

问题改写是把原始问题改成更清楚、更适合检索的说法，解决口语、省略、指代和术语不一致造成的召回问题。单轮问题可以删除无意义的连接词、补全省略的主语；多轮对话则要把历史中明确的指代对象合并到当前问题中。改写只应使用用户问题和已授权的对话历史，不能把预期答案、目标页码或知识库原文偷偷写进检索词。

改写的优点是成本低、容易接入现有检索器，并能让检索问题和文档使用相近的词汇；风险是模型可能改变问题范围、加入用户没有问的事实，或把一个本来已经找全的问题改坏。因此应保留原问题并比较改写前后覆盖率，只有回答要点或目标资料确实改善时才采用改写结果。

带对话历史时，没有历史就原样检索；有历史时，先让模型把当前追问改写成独立问题，再交给同一个检索器。下面使用教程统一的 `glm-4-flash` 调用方式。

```python
from common.nontraining_utils import llm_call

history = '用户：我刚看了《哪吒之魔童降世》。'
question = '这部电影的导演是谁？'
standalone_question = llm_call(
    '根据历史对话把当前问题改成独立问题。只输出改写后的问题，不要回答。\n'
    f'历史：{history}\n当前问题：{question}'
)
retrieved_docs = search(standalone_question, top_k=8)
```

In [1]:
import sys
from pathlib import Path


def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

In [2]:
from common.eval_utils import emit_tutorial_audit

import json
from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages, page_coverage
from common.nontraining_utils import load_annotation

cases = {item["id"]: item for item in load_query_catalog()}
search = build_bm25_search(load_pdf_pages())


def target_rank(results, expected_pages):
    expected = set(expected_pages)
    return next((rank for rank, item in enumerate(results, start=1) if item.page in expected), None)


def context_chars(results):
    return sum(len(item.text) for item in results)


def limit_context(results, char_limit):
    limited = []
    remaining = char_limit
    for item in results:
        text = item.text[:max(remaining, 0)]
        limited.append(type(item)(item.page, text, item.score))
        remaining -= len(text)
    return limited

def standard_metrics(results, expected_pages):
    pages = [int(item.page) for item in results]
    expected = set(int(page) for page in expected_pages)
    found = {page for page in pages if page in expected}
    first = next((rank for rank, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': first, 'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def emit_standard(method, role, case_id, before, after, expected_pages, check_purpose=None):
    payload = {'case_id': case_id, 'method': method, 'role': role,
               'before': standard_metrics(before, expected_pages),
               'after': standard_metrics(after, expected_pages)}
    if check_purpose:
        payload['check_purpose'] = check_purpose
    emit_tutorial_audit(payload)


## 单次改写、MultiQuery 与 decomposition 的边界

这三个操作都发生在检索前，但解决的不是同一个问题：

| 操作 | 查询形态 | 适用条件 | 风险 | 结果合并 |
| --- | --- | --- | --- | --- |
| 单次 query rewrite | 把一个原问题改成一个更清楚的查询 | 口语、省略、指代或术语不一致；仍是一个意图 | 改变范围、加入未问事实；可能把已找全的问题改坏 | 原问题与改写结果二选一或对照，通常只保留一条查询的结果 |
| MultiQuery | 为同一个意图生成多个不同措辞的查询 | 同一目标可能有术语、同义词或词序差异；不是把问题拆成几件事 | 查询数增加、重复和噪声增加；模型可能偷偷改变意图 | 原问题也必须检索；按稳定文档 ID 首次出现顺序去重，再限制候选数量 |
| decomposition（拆题） | 一个复杂问题拆成多个独立子目标 | 问题确实要求多个互补方面，且每个子目标都要有证据 | 丢条件、改变逻辑关系或漏掉子目标；检索和回答成本更高 | 按子目标保留命中来源，去重后合并，并在回答前检查每个子目标是否覆盖 |

因此，MultiQuery 的多条查询应当仍然能回答同一个问题；decomposition 的多条查询分别回答不同子目标。前者扩大同一意图的表达覆盖，后者扩大问题组成的覆盖，不能把前者的 variants 叫作子问题，也不能把一次结果的变化说成普遍提升。下面只做一个最小真实闭环：模型生成一次结构化 variants，随后原问题和 variants 都交给同一个本地检索器。模型输出、每条查询的命中和按稳定文档 ID 合并后的结果都会保存在审计记录中。

代码只对 JSON 结构、数量、非空、去重、与原问题的词汇交集和单问题形态做保守闸门；这些检查不能证明语义等价，生产系统仍需用开发集和人工抽查确认没有漂移。

In [3]:
import os
import re
from dotenv import dotenv_values
from zhipuai import ZhipuAI

# 只从项目根目录 .env 读取密钥；缺失、依赖或 API 响应错误都直接失败。
multiquery_env_path = course_root.parents[1] / ".env"
if not multiquery_env_path.is_file():
    raise FileNotFoundError(f"没有找到项目根目录 .env：{multiquery_env_path}")
multiquery_env_values = dotenv_values(multiquery_env_path, interpolate=False)
multiquery_api_key = multiquery_env_values.get("ZHIPUAI_API_KEY")
if not multiquery_api_key:
    raise RuntimeError(".env 没有设置 ZHIPUAI_API_KEY，无法执行真实 MultiQuery 实验")

MULTIQUERY_MODEL = "glm-4-flash"
MULTIQUERY_VARIANT_COUNT = 3
MULTIQUERY_TOP_K = 3
multiquery_case = cases["cross_validation_reliability"]
multiquery_original_query = str(multiquery_case["query"]).strip()
if not multiquery_original_query:
    raise ValueError("MultiQuery 原问题不能为空")

multiquery_prompt = f"""
你是检索查询改写器。给定一个用户问题，请生成恰好 {MULTIQUERY_VARIANT_COUNT} 条不同措辞的查询。
它们必须保持同一个回答目标、实体、比较关系和限定条件；每条仍是一个完整问题，不能拆成多个子问题，不能回答问题，不能加入原问题没有的事实。
只输出合法 JSON 对象，不要 Markdown 围栏或其他文字，格式必须是：
{{\"variants\":[\"查询一\",\"查询二\",\"查询三\"]}}\n用户问题：{multiquery_original_query}
"""
multiquery_client = ZhipuAI(api_key=multiquery_api_key, max_retries=0)
multiquery_response = multiquery_client.chat.completions.create(
    model=MULTIQUERY_MODEL,
    messages=[{"role": "user", "content": multiquery_prompt}],
    temperature=0.0,
    max_tokens=300,
    timeout=60,
)
multiquery_raw_output = multiquery_response.choices[0].message.content
if not isinstance(multiquery_raw_output, str) or not multiquery_raw_output.strip():
    raise RuntimeError("glm-4-flash 返回空的 MultiQuery JSON")

# 严格 JSON：不接受围栏、前后解释或静默修复。
multiquery_payload = json.loads(multiquery_raw_output)
if not isinstance(multiquery_payload, dict):
    raise ValueError("MultiQuery 模型输出必须是 JSON 对象")
multiquery_variants = multiquery_payload.get("variants")
if not isinstance(multiquery_variants, list) or len(multiquery_variants) != MULTIQUERY_VARIANT_COUNT:
    raise ValueError(f"MultiQuery 必须返回恰好 {MULTIQUERY_VARIANT_COUNT} 条 variants")
if not all(isinstance(item, str) and item.strip() for item in multiquery_variants):
    raise ValueError("每条 variant 都必须是非空字符串")

def multiquery_query_key(text):
    return re.sub(r"[\s?？。！!，,；;：:、\"'“”‘’()（）\[\]【】]", "", str(text)).casefold()

def multiquery_query_terms(text):
    return set(re.findall(r"[一-鿿]|[A-Za-z0-9]+", str(text).casefold()))

def multiquery_document_id(item):
    chunk_id = getattr(item, "chunk_id", None)
    return f"chunk:{chunk_id}" if chunk_id else f"pdf-page-{int(item.page):04d}"

multiquery_variants = [item.strip() for item in multiquery_variants]
multiquery_all_query_keys = [multiquery_query_key(multiquery_original_query)] + [multiquery_query_key(item) for item in multiquery_variants]
if len(set(multiquery_all_query_keys)) != len(multiquery_all_query_keys):
    raise ValueError("variants 必须彼此不同，且不能复制原问题")
multiquery_original_terms = multiquery_query_terms(multiquery_original_query)
for variant in multiquery_variants:
    if len(variant) > 160 or "\n" in variant or "\r" in variant:
        raise ValueError("variant 长度或格式不符合单个查询约束")
    if variant.count("？") + variant.count("?") > 1:
        raise ValueError("variant 不能包含多个问题，避免把 MultiQuery 误作 decomposition")
    if not multiquery_original_terms.intersection(multiquery_query_terms(variant)):
        raise ValueError("variant 与原问题没有词汇交集，可能发生意图漂移")

multiquery_queries = [multiquery_original_query, *multiquery_variants]
multiquery_hits_by_query = {}
for query in multiquery_queries:
    hits = search(query, top_k=MULTIQUERY_TOP_K)
    if not hits:
        raise ValueError(f"查询没有返回资料：{query}")
    multiquery_hits_by_query[query] = [
        {
            "document_id": multiquery_document_id(item),
            "page": int(item.page),
            "score": float(item.score),
            "text": item.text,
        }
        for item in hits
    ]

# 稳定 document_id 的首次出现顺序决定合并顺序；重复文档只保留一份并记录来源查询。
multiquery_merged_by_id = {}
for query in multiquery_queries:
    for hit in multiquery_hits_by_query[query]:
        document_id = hit["document_id"]
        if document_id not in multiquery_merged_by_id:
            multiquery_merged_by_id[document_id] = {**hit, "matched_queries": [query]}
        elif query not in multiquery_merged_by_id[document_id]["matched_queries"]:
            multiquery_merged_by_id[document_id]["matched_queries"].append(query)
multiquery_merged_results = list(multiquery_merged_by_id.values())
if len({item["document_id"] for item in multiquery_merged_results}) != len(multiquery_merged_results):
    raise AssertionError("合并结果的稳定 document_id 必须唯一")

print("原 query：", multiquery_original_query)
print("variants（一次 glm-4-flash 调用）：")
for index, variant in enumerate(multiquery_variants, start=1):
    print(f"  {index}. {variant}")
print("逐 query 命中：")
for query, hits in multiquery_hits_by_query.items():
    print(f"  {query}")
    print("    " + "、".join(f"{hit['document_id']}（第 {hit['page']} 页）" for hit in hits))
print("按稳定 document_id 去重后的合并结果：")
for item in multiquery_merged_results:
    print(f"  {item['document_id']}（第 {item['page']} 页），来自 {len(item['matched_queries'])} 条查询")

emit_tutorial_audit({
    "experiment": "MultiQuery 同意图多表达真实闭环",
    "model": MULTIQUERY_MODEL,
    "api_key_source": "项目根目录 .env:ZHIPUAI_API_KEY",
    "model_call_count": 1,
    "raw_model_output": multiquery_raw_output,
    "original_query": multiquery_original_query,
    "variants": multiquery_variants,
    "hits_by_query": multiquery_hits_by_query,
    "merged_results": multiquery_merged_results,
    "merge_rule": "按首次出现的稳定 document_id 去重，并保留 matched_queries",
    "scope": "单个问题的一次运行；仅展示同一意图的多表达检索，不外推普遍收益",
})

原 query： 交叉验证法为什么比单次留出法更可靠？
variants（一次 glm-4-flash 调用）：
  1. 为什么交叉验证法比单次留出法更可靠？
  2. 交叉验证法与单次留出法相比，为什么更可靠？
  3. 单次留出法与交叉验证法相比，为什么交叉验证法更可靠？
逐 query 命中：
  交叉验证法为什么比单次留出法更可靠？
    pdf-page-0019（第 19 页）、pdf-page-0018（第 18 页）、pdf-page-0026（第 26 页）
  为什么交叉验证法比单次留出法更可靠？
    pdf-page-0019（第 19 页）、pdf-page-0018（第 18 页）、pdf-page-0026（第 26 页）
  交叉验证法与单次留出法相比，为什么更可靠？
    pdf-page-0019（第 19 页）、pdf-page-0018（第 18 页）、pdf-page-0026（第 26 页）
  单次留出法与交叉验证法相比，为什么交叉验证法更可靠？
    pdf-page-0019（第 19 页）、pdf-page-0018（第 18 页）、pdf-page-0026（第 26 页）
按稳定 document_id 去重后的合并结果：
  pdf-page-0019（第 19 页），来自 4 条查询
  pdf-page-0018（第 18 页），来自 4 条查询
  pdf-page-0026（第 26 页），来自 4 条查询


## 去掉口语歧义，保留问题本意

用户问“算法怎样从许多可能的函数中学出最后那个模型”。回答端只保留第一条资料时，原问题返回了无关页面。这里不把书中术语或答案塞进检索词，只删掉“是怎么”“最后那个”等口语成分，并把“学出”改成更常见的“学习”；整理后第 16 页排到第一条。

In [3]:
case = cases["algorithm_and_model"]
rewritten = case["query"].replace("是怎么", "").replace("最后那个", "").replace("学出", "学习")
rewritten = rewritten.replace("的？", "？")

# 改写只使用用户原问题中的说法；书中术语只在检索结果出来后用于复核。
retrieval_calls_before = 1
retrieval_calls_after = 1
before_raw = search(case["query"], top_k=5)
after_raw = search(rewritten, top_k=5)
annotation = load_annotation(case['id'])
direct_context_cap = min(context_chars(before_raw), context_chars(after_raw))
before = limit_context(before_raw, direct_context_cap)
after = limit_context(after_raw, direct_context_cap)
required_points = (("假设空间", "一元一次函数"), ("机器学习算法", "学得模型"))

def answer_point_coverage(results, use_first_only=True):
    selected = results[:1] if use_first_only else results
    text = "".join(item.text.replace(" ", "") for item in selected)
    checks = [all(part in text for part in point) for point in required_points]
    return sum(checks) / len(checks), checks

before_points, before_checks = answer_point_coverage(before)
after_points, after_checks = answer_point_coverage(after)
before_first_supports_answer = before_points == 1.0
after_first_supports_answer = after_points == 1.0

print("原问题：", case["query"])
print("改写后：", rewritten)
print("原结果页：", [item.page for item in before])
print("改写后结果页：", [item.page for item in after])
print("目标页排名：", target_rank(before, annotation["expected_pages"]), "→", target_rank(after, annotation["expected_pages"]))
print("第一条资料的回答要点覆盖率：", before_points, "→", after_points)
print("第一条资料的两项检查：", before_checks, "→", after_checks)
print("返回资料量上限：5 → 5；实际返回条数：", len(before), "→", len(after))
print("字符上限（两边相同）：", direct_context_cap, "；实际上下文字符数：", context_chars(before), "→", context_chars(after))
print("实际送入回答的资料量：1 → 1；检索次数：", retrieval_calls_before, "→", retrieval_calls_after)

assert context_chars(before) == context_chars(after) == direct_context_cap
assert not before_first_supports_answer and after_first_supports_answer and before_points == 0.0 and after_points == 1.0
assert "假设空间" not in rewritten and "函数空间" not in rewritten and target_rank(after, annotation["expected_pages"]) < target_rank(before, annotation["expected_pages"])
emit_standard('直接改写问题', 'main', case['id'], before, after, annotation['expected_pages'])

原问题： 算法是怎么从许多可能的函数中学出最后那个模型的？
改写后： 算法从许多可能的函数中学习模型？
原结果页： [113, 18, 16, 26, 17]
改写后结果页： [16, 18, 15, 54, 109]
目标页排名： 3 → 1
第一条资料的回答要点覆盖率： 0.0 → 1.0
第一条资料的两项检查： [False, False] → [True, True]
返回资料量上限：5 → 5；实际返回条数： 5 → 5
字符上限（两边相同）： 7605 ；实际上下文字符数： 7605 → 7605
实际送入回答的资料量：1 → 1；检索次数： 1 → 1



## 拆成子问题

算法参数（超参数）和模型参数有什么区别？以支持向量机为例，C、w、b 分别属于哪一类？这是同一问题中的两个分类问法。原问题直接取两条资料时会漏掉第 19 页的定义；拆成两个只由用户原话组成的短问题后，各取一条并去重，合并结果找回 C 与 w、b 的两段说明。两个子问题不加入答案词、原文片段或预期页码；最终仍最多保留两条资料。

拆成子问题采用分而治之：先由规则或语言模型把复杂问题拆成若干更具体的问题，分别检索，再合并、去重和压缩上下文，再回答原问题。它适合一个问题确实包含多个互相独立的方面，或一个长问题的词汇会掩盖其中某一项；优点是每一项能使用更匹配的检索词，回答覆盖也更完整。代价是每个子问题都可能增加一次检索和一次模型调用，还可能重复返回同一片段、造成上下文膨胀；拆分质量差时会改变原问题或丢失条件，所以要限制子问题数并保留原问题作最终核对。

下面是一个不依赖特定知识库的代码写法。当前 Notebook 的实验使用人工写出的、只由原问题词语组成的子问题并检查去重；如果让 LLM 自动生成子问题，需要额外验证它没有引入答案词、页码或用户没有提出的条件。

```python
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

split_prompt = ChatPromptTemplate.from_template(
    '把下面复杂问题拆成最多 3 个独立子问题，每行一个，不要回答：\n{question}'
)
split_chain = split_prompt | llm | StrOutputParser()
def retrieve_subqueries(question):
    queries = [line.strip() for line in split_chain.invoke({'question': question}).splitlines()]
    docs = [doc for query in queries for doc in retriever.invoke(query)]
    return deduplicate(docs)
result = RunnableLambda(lambda x: retrieve_subqueries(x['question']))
```

In [4]:
case = cases["hyperparameter_vs_parameter"]
subqueries = [
    # 两个子问题只由原问题中的词重排而成，不加入答案词或页码。
    "算法参数（超参数）C",
    "模型参数 支持向量机",
]

retrieval_calls_before = 1
retrieval_calls_after = len(subqueries)
result_k = 2
before_raw = search(case["query"], top_k=result_k)
groups = [search(query, top_k=1) for query in subqueries]
after_raw = []
seen = set()
for group in groups:
    for item in group:
        key = (item.page, item.text)
        if key not in seen and len(after_raw) < result_k:
            seen.add(key)
            after_raw.append(item)
annotation = load_annotation(case['id'])
split_context_cap = min(context_chars(before_raw), context_chars(after_raw))
before = limit_context(before_raw, split_context_cap)
after = limit_context(after_raw, split_context_cap)

def hyperparameter_answer_coverage(results):
    text = "".join("".join(item.text.split()) for item in results)
    algorithm_parameter = all(term in text for term in ("算法参数", "超参数", "支持向量机", "C"))
    model_parameter = all(term in text for term in ("模型参数", "w", "b", "新样本", "预测"))
    checks = [algorithm_parameter, model_parameter]
    return sum(checks) / len(checks), checks

before_pages = page_coverage(annotation["expected_pages"], before)[1]
after_pages = page_coverage(annotation["expected_pages"], after)[1]
before_points, before_checks = hyperparameter_answer_coverage(before)
after_points, after_checks = hyperparameter_answer_coverage(after)
before_chars = sum(len(item.text) for item in before)
after_chars = sum(len(item.text) for item in after)
before_rank = target_rank(before, annotation["expected_pages"])
after_rank = target_rank(after, annotation["expected_pages"])

print("主要问题：", case["query"])
print("原问题直接取两条：", [item.page for item in before])
print("拆分问题：", subqueries)
print("拆分结果页（去重后）：", [item.page for item in after])
print("必要页面覆盖率：", before_pages, "→", after_pages, "；必要页排名：", before_rank or "未命中", "→", after_rank or "未命中")
print("回答要点覆盖率：", before_points, "→", after_points, "；检查：", before_checks, "→", after_checks)
print("返回资料量上限（去重后）：2 → 2；实际返回条数：", len(before), "→", len(after))
print("字符上限（两边相同）：", split_context_cap, "；实际上下文字符数：", before_chars, "→", after_chars)
print("检索次数：", retrieval_calls_before, "→", retrieval_calls_after)

assert len(before) == len(after) == result_k and len({item.page for item in after}) == result_k
assert before_chars == after_chars == split_context_cap
assert before_points == 0.0 and after_points == 1.0 and after_rank == 1
assert before_pages == 0.0 and after_pages == 1.0
emit_standard('拆成子问题', 'main', case['id'], before, after, annotation['expected_pages'])

主要问题： 算法参数（超参数）和模型参数有什么区别？以支持向量机为例，C、w、b 分别属于哪一类？
原问题直接取两条： [104, 31]
拆分问题： ['算法参数（超参数）C', '模型参数 支持向量机']
拆分结果页（去重后）： [19, 60]
必要页面覆盖率： 0.0 → 1.0 ；必要页排名： 未命中 → 1
回答要点覆盖率： 0.0 → 1.0 ；检查： [False, False] → [True, True]
返回资料量上限（去重后）：2 → 2；实际返回条数： 2 → 2
字符上限（两边相同）： 2733 ；实际上下文字符数： 2733 → 2733
检索次数： 1 → 2



## 先查一般原理（Step-Back Prompting），再回到细节

VC 维问题同时问“增长函数反映什么”和“泛化误差界随样本数量如何变化”。原问题直接取两条资料时找到第 154 页，却没有找到第 156 页的一般结论。

这里用补查页替换首轮排序中的一条无关结果：保留首轮的第 154 页，用第 156 页替换无关的第 162 页，最终仍只给回答端两条资料。补查问题只使用用户原问题中的词，不从预期页码、原文片段或参考回答取词；它不是把原问题改成简单问题，而是先补齐回答细节所需的一般原理。它和“按首轮概念补查”不同：这里的补查词在第一次检索前就能从问题确定，后者则必须等首轮结果实际出现一个概念。

Step-Back Prompting 是先从具体问题后退一步，生成一个更抽象、更通用的问题，再把抽象问题和原问题都用于检索。它由 Google DeepMind 的 [Take a Step Back: Evoking Reasoning via Abstraction in Large Language Models](https://arxiv.org/pdf/2310.06117) 提出。抽象问题用于找一般定义、规律或原理，原问题用于找具体事实，二者的资料一起交给回答模型。

![Step-Back Prompting](./figures/Step-Back-Prompting.png)

它适合细节问题缺少背景、问题同时要求一般规律和具体结论，或文档把原理与例子分在不同位置的情况；通过‘抽象化 → 具体推理’可以减少模型直接在细节上迷路。它不是所有问题都更好：额外的生成和检索会增加延迟，抽象得太宽会带来噪声，效果也依赖模型的领域知识和抽象能力。若原问题已经找全，或问题包含必须保持不变的编号、日期和权限条件，就不应强行后退。

下面只说明 Few-shot（给模型几个示例）如何把具体问题改成更一般的问题。`DuckDuckGoSearchAPIWrapper` 是外部搜索服务；本页实验使用随附的《南瓜书》，因此这里不把外部调用当作本教程结果。

```python
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

examples = [
    {'input': '中国农历年腊月三十日放假吗？', 'output': '中国农历假期时间有哪些？'},
    {'input': '李白是哪个朝代的诗人？', 'output': '李白的生平简介？'},
]
example_prompt = ChatPromptTemplate.from_messages([('human', '{input}'), ('ai', '{output}')])
few_shot = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=examples)
step_back_prompt = ChatPromptTemplate.from_messages([
    ('system', '把具体问题改写成更抽象、更通用的问题，不要回答。'),
    few_shot, ('user', '{question}'),
])
question_gen = step_back_prompt | llm | StrOutputParser()
search = DuckDuckGoSearchAPIWrapper(max_results=4)
# normal_context = search.run(question)
# step_back_context = search.run(question_gen.invoke({'question': question}))
```

In [5]:
case = cases["vc_dimension_generalization_bound"]
general_query = "VC 维 泛化误差界"

retrieval_calls_before = 1
retrieval_calls_after = 2
before_raw = search(case["query"], top_k=2)
general_result = search(general_query, top_k=1)
annotation = load_annotation(case['id'])
# 用补查到的第 156 页替换首轮中的无关第 162 页，保留首轮第 154 页。
after_raw = [general_result[0], before_raw[1]]
general_context_cap = min(context_chars(before_raw), context_chars(after_raw))
before = limit_context(before_raw, general_context_cap)
after = limit_context(after_raw, general_context_cap)

def vc_answer_coverage(results):
    text = "".join("".join(item.text.split()) for item in results)
    growth_function = "增长函数" in text and "复杂度" in text
    generalization_bound = "收敛速率" in text and "样本数量m" in text
    checks = [growth_function, generalization_bound]
    return sum(checks) / len(checks), checks

before_coverage = page_coverage(annotation["expected_pages"], before)[1]
after_coverage = page_coverage(annotation["expected_pages"], after)[1]

print("原问题结果页：", [item.page for item in before])
print("一般原理结果页：", [item.page for item in general_result])
print("必要页面覆盖率：", before_coverage, "→", after_coverage)
before_points, before_checks = vc_answer_coverage(before)
after_points, after_checks = vc_answer_coverage(after)
print("回答要点覆盖率：", before_points, "→", after_points)
print("两项回答检查：", before_checks, "→", after_checks)
print("返回资料量上限：2 → 2；实际返回条数：", len(before), "→", len(after), "；去重后页数：", len({item.page for item in before}), "→", len({item.page for item in after}))
print("字符上限（两边相同）：", general_context_cap, "；实际上下文字符数：", context_chars(before), "→", context_chars(after))
print("检索次数：", retrieval_calls_before, "→", retrieval_calls_after)

assert general_result[0].page == 156 and len(before) == len(after) == 2
assert context_chars(before) == context_chars(after) == general_context_cap
assert before_points == before_coverage == 0.5 and after_points == after_coverage == 1.0
emit_standard('先查一般原理', 'main', case['id'], before, after, annotation['expected_pages'], '再次改善')

原问题结果页： [162, 154]
一般原理结果页： [156]
必要页面覆盖率： 0.5 → 1.0
回答要点覆盖率： 0.5 → 1.0
两项回答检查： [True, False] → [True, True]
返回资料量上限：2 → 2；实际返回条数： 2 → 2 ；去重后页数： 2 → 2
字符上限（两边相同）： 2690 ；实际上下文字符数： 2690 → 2690
检索次数： 1 → 2



## 换一道题：原问题已经找全时不必改写

“哪些模型都能把训练集拟合好时，这批模型叫什么？”在当前全文检索中已经把第 16 页排在第一条，而且第一条含有两项必要说明。这里仍做一次只去掉口语连接的改写；没有改善就如实记录。直接改写真正有效的是前面的算法与模型问题，因为它在改写前确实答不全。

In [6]:
case = cases["version_space_definition"]
rewritten = case["query"].replace("都能", "能够").replace("时，这批", "？这批")
before_raw = search(case["query"], top_k=1)
after_raw = search(rewritten, top_k=1)
annotation = load_annotation(case['id'])
version_context_cap = min(context_chars(before_raw), context_chars(after_raw))
before = limit_context(before_raw, version_context_cap)
after = limit_context(after_raw, version_context_cap)

def version_space_points(results):
    text = "".join("".join(item.text.split()) for item in results[:1])
    checks = ["拟合训练集" in text and "模型" in text, "版本空间" in text]
    return sum(checks) / len(checks), checks

before_points, before_checks = version_space_points(before)
after_points, after_checks = version_space_points(after)
print("原问题：", case["query"])
print("改写后：", rewritten)
print("改前结果页：", [item.page for item in before], "；改后结果页：", [item.page for item in after])
print("目标页排名：", target_rank(before, annotation["expected_pages"]), "→", target_rank(after, annotation["expected_pages"]))
print("第一条资料的回答要点覆盖率：", before_points, "→", after_points, "；检查：", before_checks, "→", after_checks)
print("返回资料量上限：1 → 1；实际返回条数：", len(before), "→", len(after), "；检索次数：1 → 1")
print("字符上限（两边相同）：", version_context_cap, "；实际上下文字符数：", context_chars(before), "→", context_chars(after))
print("明确结论：原问题已经找全，改写没有增加覆盖；真正有效的是前面的算法与模型问题。")

assert before_points == after_points == 1.0 and target_rank(before, annotation["expected_pages"]) == target_rank(after, annotation["expected_pages"]) == 1 and "版本空间" not in rewritten
assert context_chars(before) == context_chars(after) == version_context_cap
emit_standard('直接改写问题', 'check', case['id'], before, after, annotation['expected_pages'], '说明不适用或限制')

原问题： 哪些模型都能把训练集拟合好时，这批模型叫什么？
改写后： 哪些模型能够把训练集拟合好？这批模型叫什么？
改前结果页： [16] ；改后结果页： [16]
目标页排名： 1 → 1
第一条资料的回答要点覆盖率： 1.0 → 1.0 ；检查： [True, True] → [True, True]
返回资料量上限：1 → 1；实际返回条数： 1 → 1 ；检索次数：1 → 1
字符上限（两边相同）： 1492 ；实际上下文字符数： 1492 → 1492
明确结论：原问题已经找全，改写没有增加覆盖；真正有效的是前面的算法与模型问题。



## 二次检查：拆成子问题的适用范围

代价敏感错误率的问题也可以按用户原话拆成“正例和反例错误率怎样组成”和“costse 怎样归一化”两问。本题直接取两条资料已经包含两项回答；拆分后仍只保留两条，并施加相同字符上限后比较实际字符数，回答内容没有增加，因此说明这种问题不必为了形式拆问。

In [7]:
case = cases["cost_sensitive_error_components"]
subqueries = [
    "正例和反例错误率组成代价敏感错误率",
    "costse 归一化时要除以什么？",
]
retrieval_calls_before = 1
retrieval_calls_after = len(subqueries)
cost_result_k = 2
before_raw = search(case["query"], top_k=cost_result_k)
groups = [search(query, top_k=1) for query in subqueries]
after_raw = []
seen = set()
for group in groups:
    for item in group:
        key = (item.page, item.text)
        if key not in seen and len(after_raw) < cost_result_k:
            seen.add(key)
            after_raw.append(item)
annotation = load_annotation(case['id'])
cost_context_cap = min(context_chars(before_raw), context_chars(after_raw))
before = limit_context(before_raw, cost_context_cap)
after = limit_context(after_raw, cost_context_cap)

def cost_points(results):
    text = "".join("".join(item.text.split()) for item in results)
    checks = ["FNR" in text and "FPR" in text, "max(costse)" in text and "除以" in text]
    return sum(checks) / len(checks), checks

before_points, before_checks = cost_points(before)
after_points, after_checks = cost_points(after)
before_pages = page_coverage(annotation["expected_pages"], before)[1]
after_pages = page_coverage(annotation["expected_pages"], after)[1]
before_chars = sum(len(item.text) for item in before)
after_chars = sum(len(item.text) for item in after)

print("原问题结果页：", [item.page for item in before])
print("拆分问题：", subqueries)
print("拆分结果页（去重后）：", [item.page for item in after])
print("必要页面覆盖率：", before_pages, "→", after_pages)
print("回答要点覆盖率：", before_points, "→", after_points, "；检查：", before_checks, "→", after_checks)
print("返回资料量上限（去重后）：2 → 2；实际返回条数：", len(before), "→", len(after))
print("字符上限（两边相同）：", cost_context_cap, "；实际上下文字符数：", before_chars, "→", after_chars)
print("检索次数：", retrieval_calls_before, "→", retrieval_calls_after)
print("明确结论：在相同返回上限和字符上限下，拆分没有增加回答要点；本题不需要拆问。")

assert before_points == after_points == before_pages == after_pages == 1.0 and len(before) == len(after) == cost_result_k and [item.page for item in after] == [24, 25] and 0.8 <= after_chars / before_chars <= 1.2
assert before_chars == after_chars == cost_context_cap
emit_standard('拆成子问题', 'check', case['id'], before, after, annotation['expected_pages'], '说明不适用或限制')

原问题结果页： [25, 24]
拆分问题： ['正例和反例错误率组成代价敏感错误率', 'costse 归一化时要除以什么？']
拆分结果页（去重后）： [24, 25]
必要页面覆盖率： 1.0 → 1.0
回答要点覆盖率： 1.0 → 1.0 ；检查： [True, True] → [True, True]
返回资料量上限（去重后）：2 → 2；实际返回条数： 2 → 2
字符上限（两边相同）： 3573 ；实际上下文字符数： 3573 → 3573
检索次数： 1 → 2
明确结论：在相同返回上限和字符上限下，拆分没有增加回答要点；本题不需要拆问。



## 二次检查：先查一般原理（线性判别分析）

线性判别分析问题的首轮结果有较大广义特征值所在的第 44 页，却没有投影目标所在的第 41 页。先用问题本身的词查一次投影的一般原理，再用补查页替换首轮最后一条无关结果，保留其余首轮资料。补查内容只根据用户问题确定，不使用答案页码或参考答案。

In [8]:
case = cases["lda_goal_and_eigenvector"]
general_query = "投影后的同类和异类样本呈现什么关系？"
before_raw = search(case["query"], top_k=5)
general_result = search(general_query, top_k=1)
annotation = load_annotation(case['id'])
# 用补查页替换首轮最后一条无关结果，避免把“保留全部原结果”误写成结论。
after_raw = [*before_raw[:4], *general_result]
lda_context_cap = min(context_chars(before_raw), context_chars(after_raw))
before = limit_context(before_raw, lda_context_cap)
after = limit_context(after_raw, lda_context_cap)

def lda_points(results):
    text = "".join("".join(item.text.split()) for item in results)
    checks = ["同类样本" in text and "很相近" in text and "异类样本" in text and "很疏远" in text, "N−1个最大的" in text and "广义特征值" in text and "特征向量" in text]
    return sum(checks) / len(checks), checks

before_points, before_checks = lda_points(before)
after_points, after_checks = lda_points(after)
before_pages = page_coverage(annotation["expected_pages"], before)[1]
after_pages = page_coverage(annotation["expected_pages"], after)[1]
print("首轮结果页：", [item.page for item in before])
print("一般原理检索词：", general_query, "；结果页：", [item.page for item in general_result])
print("改后保留页：", [item.page for item in after])
print("必要页面覆盖率：", before_pages, "→", after_pages)
print("回答要点覆盖率：", before_points, "→", after_points, "；检查：", before_checks, "→", after_checks)
print("返回资料量上限：5 → 5；实际返回条数：", len(before), "→", len(after), "；检索次数：1 → 2")
print("字符上限（两边相同）：", lda_context_cap, "；实际上下文字符数：", context_chars(before), "→", context_chars(after))
print("明确结论：先查投影的一般原理补回第 41 页，两项回答要点均找全。")

assert general_result[0].page == 41 and before_points == before_pages == 0.5 and after_points == after_pages == 1.0 and len(before) == len(after) == 5
assert context_chars(before) == context_chars(after) == lda_context_cap
emit_standard('先查一般原理', 'check', case['id'], before, after, annotation['expected_pages'], '再次改善')

首轮结果页： [59, 14, 44, 181, 126]
一般原理检索词： 投影后的同类和异类样本呈现什么关系？ ；结果页： [41]
改后保留页： [59, 14, 44, 181, 41]
必要页面覆盖率： 0.5 → 1.0
回答要点覆盖率： 0.5 → 1.0 ；检查： [False, True] → [True, True]
返回资料量上限：5 → 5；实际返回条数： 5 → 5 ；检索次数：1 → 2
字符上限（两边相同）： 6918 ；实际上下文字符数： 6918 → 6918
明确结论：先查投影的一般原理补回第 41 页，两项回答要点均找全。



## 怎么判断是否值得改写

直接改写把算法与模型问题的目标页从第 3 位提到第 1 位；拆成两问让算法参数 C 与模型参数 w、b 的定义在相同两条资料预算内找回第 19 页，同时没有增加最终资料数量；先查一般原理也补全了两道题的回答要点。换题检查时，版本空间和代价敏感错误率问题原本已经答全：前者没有改善，后者在相同资料预算下也没有增加回答内容。

如果改写后没有更接近目标资料，或者引入了用户没问的新内容，就继续使用原问题。